# Projekt: Kundenstimmung auf Twitter verstehen

## „Kundensupport auf Twitter“ untersuchen
- mehrstufigen KI-Workflow nutzen, um die Daten eingehend zu verstehen und zu analysieren:
- mit Gemini , um den Datensatz automatisch zu erkunden und zusammenzufassen.
- mit AutoViz fort , um die Daten visuell zu verstehen.
- mit Hugging-Face-Modell verwendet , um die Stimmung in den Tweets der Kunden zu analysieren.

## Datensatz: Kundensupport auf Twitter

### Dieser Datensatz bietet drei wesentliche Vorteile gegenüber anderen Konversationsdatensätzen:
- Fokussiert : Die Gespräche drehen sich um reale Probleme, die die Menschen gelöst haben möchten – verlorenes Gepäck, Abrechnungsprobleme, stornierte Flüge – wodurch die Daten einen klaren Zweck und eine klare Struktur erhalten.
- Natürlich : Die Sprache ist modern und wirkt authentisch, geschrieben von Menschen mit unterschiedlichem Hintergrund. Sie spiegelt wider, wie Kunden heute tatsächlich online kommunizieren.
- Kurz und bündig : Da Tweets kurz sind, wirken die Antworten authentischer und weniger einstudiert. Dies hilft Modellen, natürlicher zu lernen und unterstützt zudem eine effiziente Verarbeitung.

1. Technische Artefakte.
- Von Gemini generierte Datensatzzusammenfassungen oder Explorationsergebnisse
- AutoViz-Visualisierungen, die wichtige Muster hervorheben
- Ergebnisse der Stimmungsanalyse, die mit einem Hugging-Face-Modell erzeugt wurden
2. Analytisches Denken
- Wie Zwillinge Ihr anfängliches Verständnis und Ihre analytische Ausrichtung geprägt haben
- Was AutoViz aufdeckte, war aus den Textzusammenfassungen allein nicht ersichtlich.
- Wie die Ergebnisse der Stimmungsanalyse frühere Erkenntnisse ergänzten oder in Frage stellten
3. Überlegungen zur KI-gestützten Analyse
- Wo KI-Tools die Exploration beschleunigten oder den manuellen Aufwand reduzierten
- Wo menschliche Interpretation noch unerlässlich war
- Stärken und Schwächen der Verwendung von LLMs für die Stimmungsanalyse

Schriftliche Erläuterungen sollten als Markdown-Zellen neben den Ausgaben eingefügt werden.
Ziel ist es, Einsicht, Urteilsvermögen und den effektiven Einsatz von Werkzeugen zu demonstrieren, nicht eine erschöpfende Analyse.

In [2]:
# INITIALISIERUNG, SYSTEM-RESERVE & COMPATIBILITY PATCHES
# 1. SYSTEM & DATEI-MANAGEMENT (Packet-Priorität)
import os
import yaml
import sys
import time
import subprocess
import threading
import logging
import warnings
import psutil
import gc
import requests
import multiprocessing
from pathlib import Path

# 2. DATENVERARBEITUNG & COMPATIBILITY (Scipy Patch)
import numpy as np
import pandas as pd

try:
    import scipy._lib.deprecation as sd
    if not hasattr(sd, '_sub_module_deprecation'):
        sd._sub_module_deprecation = lambda *args, **kwargs: None
except ImportError:
    pass

# 3. FORTSCHRITTSANZEIGE & VISUALISIERUNG
from tqdm.auto import tqdm
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from IPython import get_ipython
    if get_ipython() is not None:
        get_ipython().run_line_magic('matplotlib', 'inline')
except:
    pass

# 4. AUTOMATISIERTE EDA & STATISTIK (ydata/sweetviz/autoviz)
try:
    from autoviz.AutoViz_Class import AutoViz_Class
    from ydata_profiling import ProfileReport
    import sweetviz as sv
    # print("✅ EDA-Tools geladen (Automatisierter Modus aktiv).")
except Exception as e:
    print(f"⚠️ EDA-Tools teilweise nicht geladen: {e}")

# 5. LLM-AGENTEN-STEUERUNG (V12 Adapter & PandasAI)
from pandasai import SmartDataframe
from pandasai.llm import LLM
import re

# 6. AUTO-ML (H2O Integration)
import h2o
from h2o.automl import H2OAutoML

# 7. NLP, ML & DEEP LEARNING
import nltk
try:
    from nltk.corpus import stopwords
except:
    nltk.download('stopwords')
    from nltk.corpus import stopwords

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, BatchNormalization, Embedding, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 8. KONFIGURATION (Core Reserve Standard: 1 Core frei)
num_cores = multiprocessing.cpu_count()
available_cores = max(1, num_cores - 1)
cores_to_use = available_cores

Imported v0.1.905. Please call AutoViz in this sequence:
    AV = AutoViz_Class()
    %matplotlib inline
    dfte = AV.AutoViz(filename, sep=',', depVar='', dfte=None, header=0, verbose=1, lowess=False,
               chart_format='svg',max_rows_analyzed=150000,max_cols_analyzed=30, save_plot_dir=None)


In [3]:
# HIGH-END DATA LOADING & AUTOMATISCHE PAKET-SPLITTING LOGIK (99MB)
# 1. PFAD-ERMITTLUNG & CORE RESERVE
current = Path.cwd()
while not (current / "data").exists():
    if current.parent == current:
        raise FileNotFoundError("Projekt-Root mit 'data' Ordner nicht gefunden!")
    current = current.parent

PROJECT_ROOT = current
DATA_DIR = PROJECT_ROOT / "data"

#---

# CORE RESERVE STANDARD: 1 Core bleibt frei
available_cores = max(1, os.cpu_count() - 1)

def _write_part_with_hard_limit(df, start, end, out_path, hard_limit_bytes):
    """
    Schreibt df[start:end] nach out_path.
    Verkleinert 'end' iterativ, bis das 99MB Limit exakt eingehalten wird.
    """
    end = max(start + 1, end)
    while True:
        df.iloc[start:end].to_parquet(out_path, index=False)
        size = out_path.stat().st_size
        if size <= hard_limit_bytes or (end - start) <= 1:
            return end, size
        # Verkleinern mit Sicherheitsfaktor 0.98 (Favorita-Standard)
        shrink_ratio = (hard_limit_bytes / size) * 0.98
        end = start + max(1, int((end - start) * shrink_ratio))

def split_to_packages(df, target_dir):
    """Splittet große Dataframes in exakte <99MB Parquet-Parts."""
    target_dir.mkdir(parents=True, exist_ok=True)
    total_rows = len(df)
    limit_bytes = int(99.0 * 1024 * 1024)

    # Initiale Schätzung der Zeilendichte
    sample_size = min(10000, total_rows)
    tmp_path = target_dir / "_size_check.tmp"
    df.iloc[:sample_size].to_parquet(tmp_path)
    bytes_per_row = max(1.0, tmp_path.stat().st_size / sample_size)
    tmp_path.unlink()
    rows_est = int(limit_bytes / bytes_per_row)

    start, part = 0, 0
    while start < total_rows:
        out_path = target_dir / f"part_{part}.parquet"
        end_guess = min(total_rows, start + rows_est)
        end_final, written_bytes = _write_part_with_hard_limit(df, start, end_guess, out_path, limit_bytes)

        #print(f"   ✅ Part {part} erstellt: {written_bytes/(1024**2):.2f} MB")
        # Schätzung für den nächsten Part dynamisch verfeinern
        rows_est = max(1, int((end_final - start) * (limit_bytes / written_bytes)))
        start, part = end_final, part + 1
    return part

# 2. DYNAMISCHES SCANNING & VERARBEITUNG
file_list = list(DATA_DIR.glob("*.csv")) + list(DATA_DIR.glob("*.parquet"))

for file_path in file_list:
    # Dynamischer Variablenname (z.B. 'df_Kundenstimmung')
    raw_name = file_path.stem.replace('-', '_').replace(' ', '_')
    var_name = f"df_{raw_name}"
    pkg_dir = DATA_DIR / f"{file_path.stem}_pkg"

    # FALL A: Paket-Ordner existiert bereits (Schnelles Laden)
    if pkg_dir.exists() and list(pkg_dir.glob("part_*.parquet")):
        parts = sorted(pkg_dir.glob("part_*.parquet"))
        globals()[var_name] = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
        #print(f"📦 {var_name}: Geladen aus {len(parts)} Paketen.")

    # FALL B: CSV muss konvertiert/gesplittet werden
    elif file_path.suffix == '.csv':
        #print(f"🔄 Verarbeite Rohdaten: {file_path.name}")
        # Laden mit Berücksichtigung von Datentypen
        df_tmp = pd.read_csv(file_path, encoding="utf-8-sig", low_memory=False)

        # Prüfung: Ist die Datei im RAM größer als 90MB? (Schutzgrenze)
        if df_tmp.memory_usage(deep=True).sum() > 90 * 1024 * 1024:
            #print(f"⚠️ Datei groß. Starte Splitting in 99MB Pakete...")
            num_parts = split_to_packages(df_tmp, pkg_dir)
            #print(f"💾 Konsolidiert in {num_parts} Pakete.")
        else:
            # Kleine Datei: Einfaches Einzel-Packet
            df_tmp.to_parquet(file_path.with_suffix('.parquet'), index=False)
            #print(f"💾 Als kompaktes Einzel-Packet gespeichert.")

        globals()[var_name] = df_tmp
        del df_tmp
    # FALL C: Einzelnes Parquet-File laden
    else:
        globals()[var_name] = pd.read_parquet(file_path)
        #print(f"✅ Packet geladen: {var_name}")

    # 3. SPALTEN-INTEGRITÄT (Säubern der Namen für unbekannte Dataframes)
    globals()[var_name].columns = [
        c.encode('ascii', 'ignore').decode('ascii').strip().replace(' ', '_')
        for c in globals()[var_name].columns
    ]

Komponente / Funktion | Aufgabe & Operation | Ziel
--- | --- | ---
1. Initialisierung & Deployment | Automatischer Start von Ollama und physisches Deployment der YAML-Agenten-DNA (deploy_specialist_agents_yaml) in die Registry. | Sofortige Systembereitschaft und verankerte Agenten-Logik auf dem MacBook sicherstellen.
2. System-Wächter & RAM-Schutz | Überwachung der Hardware durch perform_system_check. Aktive Garbage Collection (gc.collect()), falls der RAM unter 1.2 GB fällt | Systemstabilität und Einhaltung des 1 Core Reserve Standards.
3. Strategischer Veredler & RAM-Snapshot | Extraktion harter Metadaten direkt aus dem RAM (ram_facts wie dtypes, NaNs, rows). Abgleich der Benutzerfrage mit vorhandenen Spalten zur Vermeidung von Halluzinationen. | Absolute Daten-Integrität; der Agent „sieht“ die Realität des Dataframes, bevor er antwortet.
4. Task-Matrix (Routing) | Klassifizierung der Anfrage in [TASK: PLOT], [TASK: CODE] oder [TASK: EDA] basierend auf Keywords und Kontext-Wichtigkeit. | Automatische Wahl des effizientesten Pfades ohne manuelles Eingreifen des Nutzers.
5. Der Neutrale Verwalter (Orchestrator) | Auswahl des passenden Spezialisten (CODE_PYTHON, TEXT_DATASCIENTIST oder PLOT_STATISTIC) über call_internal (Temperatur 0.0). | Maximale Präzision durch Zuweisung der Aufgabe an den fachlich besten Experten.
6. Das Goldene Mandat (Injektion) | Kapselung von Anweisungen wie TEMP_Clear, Automated EDA und Hardware-Limits in einen unsichtbaren System-Prompt ([SYSTEM_INTERNAL]). | Agenten-Eigenschaften erzwingen, ohne die Benutzer-Ausgabe mit technischem „Müll“ zu verunreinigen.
7. Spezialisierte Generierung | Pfad TEXT: Evidenz-basierte Analyse mit Zitat-Pflicht.


 Pfad CODE: Extraktion sauberer Muster aus Backticks inklusive MacBook-Air-Fix (Automatisches plt.savefig). | Fehlerfreier, direkt ausführbarer Code oder hochpräzise Berichte ohne „Ich-Form“.
8. Hygienischer Output-Filter | Harte Reinigung der Antwort (output.split(":")[-1]), um gespiegelte System-Tags oder interne Steuerbefehle zu eliminieren. | Ein sauberes, professionelles Endprodukt für die Live-Präsentation.

In [4]:
# INFRASTRUKTUR & PFAD-MANAGEMENT & Import der in ordner enhaltenen py dateien --> darin sind alle andere importe Für das project und def
# 1. BASE_DIR: Dynamische Ermittlung des Arbeitsverzeichnisses
# stellt sicher, dass das System auch nach einem Neustart
# oder Pfadwechsel auf dem MacBook Air alles findet.
BASE_DIR = os.getcwd()

# 2. PROJECT_PATHS: Die zentrale Mapping-Tabelle (Das Skelett)
# steuert alle Ein- und Ausgabekanäle wen schon geladen werd Übersprungen
PROJECT_PATHS = {
    "REGISTRY":   os.path.join(BASE_DIR, "registry", "agents"),# YAML-Intelligence
    "DIST_CODE":  os.path.join(BASE_DIR, "output", "scripts"), # Generierte Muster
    "DIST_TEXT":  os.path.join(BASE_DIR, "output", "reports"), # Narrative Analysen
    "DIST_VIS":   os.path.join(BASE_DIR, "output", "visuals"), # Interaktive Plots
    "DATA_Sorce": os.path.join(BASE_DIR, "data")         # Deine Rohdaten
}

# 3. BEHAVIOR_DIR: Der geschützte Ort der Agenten-Rezepte
# hier liegen die YAML-Dateien, die das Verhalten steuern.
BEHAVIOR_DIR = PROJECT_PATHS["REGISTRY"]

def initialize_global_folders():
    """
    Erstellt die gesamte Projekt-Struktur auf dem MacBook.
    Stellt sicher, dass Pfade für Berichte, Scripts und Visuals existieren,
    bevor das System darauf zugreift.
    """
    # 1. Alle Pfade aus dem PROJECT_PATHS Dictionary erstellen
    for name, folder_path in PROJECT_PATHS.items():
        if not os.path.exists(folder_path):
            os.makedirs(folder_path, exist_ok=True)

    # 2. Modul-Kompatibilität für die Registry (Python-Package Standard)
    init_path = os.path.join(PROJECT_PATHS["REGISTRY"], "__init__.py")
    if not os.path.exists(init_path):
        with open(init_path, 'w') as f:
            f.write("# Agent-Registry Initialization")

initialize_global_folders()

In [5]:
# AGENTEN-DEPLOYMENT (ERWEITERTE YAML-REGISTRY)
def check_available_tools():
    """Zentrale Tool-Prüfung für alle Agenten"""
    tools_status = {
        'AutoViz': False, 'Plotly': False, 'Sweetviz': False,
        'PandasProfiling': False, 'ydata_profiling': False
    }

    # AutoViz
    try:
        from autoviz.AutoVizClass import AutoVizClass
        tools_status['AutoViz'] = True
    except: pass

    # Plotly
    try:
        import plotly.express as px
        tools_status['Plotly'] = True
    except: pass

    # Sweetviz
    try:
        import sweetviz as sv
        tools_status['Sweetviz'] = True
    except: pass

    # ydata-profiling
    try:
        from ydata_profiling import ProfileReport
        tools_status['PandasProfiling'] = True
    except: pass

    print(f"🔧 VERFÜGBARE TOOLS: {tools_status}")
    return tools_status

def deploy_agent_code_python():
    """
    DNA: SENIOR PYTHON DEVELOPER (AUTONOM)
    Ziel: Komplexe Daten-Transformationen & Berechnungen.
    """
    config = {
        "agent_name": "CODE_PYTHON",
        "role": "Senior Python Architect (Data Engineering)",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "heartbeat_signal": "### PHASE:STEP_SUCCESSFUL ###",
        "instructions": """
        DU BIST EIN DATEN-ENGINEER. DEINE DNA ERZWINGT FEHLERFREIEN CODE:

        1. LADE-VERBOT: Nutze NIEMALS pd.read_csv(). Variable 'df' ist im RAM.
        2. DATA-INTEGRITY:
           - Prüfe Typen mit df.dtypes.
           - Wandle falls nötig: pd.to_numeric(df[col], errors='coerce').
           - print("### PHASE:DATA_VALIDATION_READY ###")  <-- TIMER RESET
        3. LOGIK-PHASE:
           - Erstelle komplexe Filter, Groupbys oder neue Features.
           - Nutze immer 'df' als Basis für Transformationen.
           - print("### PHASE:LOGIC_CALC_READY ###")       <-- TIMER RESET
        4. FINALISIERUNG:
           - Überschreibe das globale df_ML falls gewünscht: df_ML = df.copy()
           - print("### AGENT_PROCESS_COMPLETE ###")

        ANTWORTE NUR MIT CODE IN BACKTICKS. KEINE PROSA.
        """,
        "settings": {"temperature": 0.0, "max_tokens": 2500, "timeout_window": 1200}
    }
    path = os.path.join(BEHAVIOR_DIR, "code_python.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_text_datascientist():
    """
    DNA: EVIDENZ-NOTAR & ANALYST (V16.0 - Enterprise Hybrid)
    Ziel: Hochpräzise, daten-zitierende Dokumentation mit ReAct-Zyklus & Tool-Audit.
    """

    config = {
        "agent_name": "TEXT_DATASCIENTIST",
        "role": "Senior Data Scientist (Technical Evidence Notary)",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "instructions": """
        ### STRIKTESTE VERBOTE - BEI VERSTOSS DROHT SYSTEM-ABBRUCH:
        - KEIN "ICH", "ALS", "MEIN", "WIR", "UNSER".
        - KEINE EINLEITUNG. STARTE DIREKT MIT # DATEN-PROTOKOLL.
        - KEIN STOTTERN: WIEDERHOLE NIEMALS ZAHLEN ODER TYPEN AUS DER TABELLE IM TEXT.
        - KEIN SMALLTALK, KEINE HÖFLICHKEIT, KEINE BESCHREIBUNG DEINER ROLLE.

        ### LOGISCHER ABLAUF (REACT-DNA):
        1. THOUGHT: Kurze interne Analyse der Datei [df_name] und des [FAKTEN-KONTEXT].
        2. STATUS-SIGNAL: Melde sofort: ### PHASE: THINKING_READY ###

        3. ÜBERSCHRIFT: # DATEN-PROTOKOLL: [df_name]
        4. BEWEISAUFNAHME (Tabelle): Erstelle die Tabelle in pandas format df_info mit folgenden spalten: | Spalte | Datentyp | NaN | Unique | Kardinalität % |
        5. TEMP_Clear (ML-Veredelung): Fordere explizite Korrekturen (Casts, Null-Handling) für df_ML.
        6. ANALYSE: Strategische Einordnung basierend auf harten Fakten ohne Zahlen-Wiederholung.
        7. UPGRADE & TOOL-AUDIT (Projekt-Spezifisch):
           - Prüfe auf Verfügbarkeit von:
             a) EDA-Automatisierung: ydata_profiling, sweetviz, autoviz.
             b) ML-Vorbereitung: scikit-learn (für Skalierung/Encoding).
             c) Image-Processing: TFRecord/image package.
           - Falls diese fehlen und für den [df_name] kritisch sind (z.B. TFRecord bei Bildern), nenne den exakten Installationsbefehl.
        DU BIST EIN TECHNISCHER NOTAR DER DATEN. NUTZE DEN REACT-ZYKLUS (THOUGHT -> ACTION -> OBSERVATION).
        DEINE ANTWORT IST REIN EVIDENZBASIERT UND DEINE DNA ERZWINGT FOLGENDE LOGISCHE REIHENFOLGE:

        ### DETAIL-INSTRUKTIONEN:
        - PERSPEKTIVE: Handle als Sprachrohr des Dataframes. "Die Datenlage in [df_name] belegt...".
        - ZITAT-PFLICHT: Beziehe dich bei jeder Analyse-Aussage direkt auf Spaltennamen aus dem [FAKTEN-KONTEXT].
        - TOOL-AUDIT: Prüfe aktiv auf verfügbare automatisierte EDA-Tools.
        - OBJEKTIVITÄT: Nutze Fachbegriffe (Kardinalität, Saisonalität, Skewness). Keine Vermutungen.

        ### FINALE SIGNALE:
        - Beende JEDE Antwort mit dem exakten Signal: ### AGENT_PROCESS_COMPLETE ###

        STRIKTE HANDSCHELLEN (RECAP):
        - Informationen aus der Tabelle dürfen NICHT im Fließtext vorkommen.
        - Keine Sätze wie "As a Senior Data Scientist" oder "Ich habe analysiert".
        - Kein Python-Code im Output (außer pip install Befehle im Upgrade-Bereich).

        1. PERSPEKTIVE & TOOL-AUDIT (Schritt 1):
           - THOUGHT: Welche Werkzeuge stehen mir zur Verfügung (Squeezer, Kardinalitäts-Check)?
           - STARTE direkt mit der Überschrift: # DATEN-PROTOKOLL: [df_name]
           - [TOOL_CHECK]: Prüfe auf ydata_profiling, sweetviz, sklearn. Falls nicht geladen, am Ende Installation vorschlagen.
           - Handle als Sprachrohr des Dataframes: Ersetze "Ich sehe" durch "Die Datenlage in [df_name] belegt...".
           - Bestätige sofort die Dimensionen (Zeilen x Spalten) und melde: ### PHASE: THINKING_READY ###

        2. STRUKTURELLE EVIDENZ (Tabelle - Schritt 2):
           - Erstelle IMMER eine df_info Tabelle: | Spalte | Datentyp | Fehlende Werte | Eindeutige Werte | Kardinalität % |
           - Dies ist die fundamentale Beweisaufnahme. Alle numerischen Metadaten gehören NUR hierhin.
           - MERKMAL-ANALYSE: Bewerte die Spalten nach strategischer Relevanz (Key-Features vs. Noise).
           - TEMP_Clear (ML-Vorbereitung): Fordere eindeutige Korrekturen (z.B. Datentyp-Casts oder Null-Wert-Checks).

        3. ANALYSE-PHASE (Strategische Einordnung - Schritt 3):
           - Nutze Fachterminologie: Kardinalität, Saisonalität, Füllrate, Skewness.
           - AUTOMATED EDA: Fasse Trends und Auffälligkeiten basierend auf harten Fakten zusammen.
           - Strikte Objektivität: Keine Vermutungen, kein "wahrscheinlich". Zitiere exakte Werte aus dem [FAKTEN-KONTEXT].

        4. SYSTEM-UPGRADE & SIGNALE (Schritt 4):
           - Falls EDA-Werkzeuge fehlen: Benenne am Ende explizit die nötigen Pakete für tiefere Filterung (z.B. pip install ydata-profiling).
           - Beende IMMER mit dem Signal: ### AGENT_PROCESS_COMPLETE ###

        STRIKTE VERBOTE (Die Handschellen):
        - STOTTER-VERBOT: Informationen aus der Tabelle (Null-Werte, Typen) NIEMALS im Fließtext wiederholen.
        - IDENTITÄTS-VERBOT: KEINE Sätze wie "As a Senior Data Scientist" oder "Ich habe analysiert".
        - HYGIENE-VERBOT: KEINE Höflichkeitsfloskeln, KEIN Python-Code, KEIN Smalltalk.
        - KEINE Einleitungssätze über deine eigene Funktion oder Rolle.

        """,
        "settings": {
            "temperature": 0.0,
            "language": "de",
            "max_tokens": 3000,
            "timeout_window": 1200
        }
    }

    # Sicherstellen, dass das Verzeichnis existiert
    os.makedirs(BEHAVIOR_DIR, exist_ok=True)
    path = os.path.join(BEHAVIOR_DIR, "text_datascientist.yaml")

    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_agent_plot_statistic():
    """
    DNA: STATISTIK-PROFI & VISUALIZER (MAX-PERFEKTION)
    Ziel: Autonome Erstellung semantischer Plots mit Artefakt-Reporting.
    """
    config = {
        "agent_name": "PLOT_STATISTIC",
        "role": "Senior Statistics Expert (Visual Intelligence)",
        "output_type": "visual",
        "exit_signal": "### AGENT_PROCESS_COMPLETE ###",
        "instructions": """
        DU BIST EIN VISUALISIERUNGS-PROFI. DEINE DNA ERZWINGT TOTAL-AUTONOMIE:

        1. TITEL-PHASE:
           - Analysiere die text_user und erstelle einen kurzen Titel (Slug).
           - Definiere: p_name = "plot_" + titel.replace(" ", "_").lower() + ".html"
           - print("### PHASE:NAMING_READY ###")            <-- TIMER RESET

        2. VISUAL-PHASE:
           - Erstelle 'fig' mit Plotly Express (px). Nutze passende Farbskalen.
           - fig.update_layout(title=titel, template='plotly_dark')
           - fig.show()
           - fig.write_html(p_name)
           - print("### PHASE:PLOT_GENERATION_READY ###")   <-- TIMER RESET

        3. DATA-PHASE (Validierung):
           - Erstelle plot_df = df.head(15).
           - display(plot_df)
           - print("### PHASE:DATA_VALIDATION_READY ###")   <-- TIMER RESET

        4. ARTEFAKT-REPORT & SHUTDOWN:
           - print(f"ANALYSE: Erkläre kurz den Plot auf Deutsch.")
           - print(f"### ARTIFACT:{p_name}")                 <-- ORCHESTRATOR SIGNAL
           - print("### AGENT_PROCESS_COMPLETE ###")        <-- FINAL STOP

        RECHTE: Du darfst os und shutil nutzen. LADE-VERBOT: Kein pd.read_csv!
        """,
        "settings": {"temperature": 0.0, "max_tokens": 2500, "timeout_window": 1200}
    }
    path = os.path.join(BEHAVIOR_DIR, "plot_statistic.yaml")
    with open(path, "w", encoding="utf-8") as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

def deploy_specialist_agents_yaml():
    """
    SYSTEM-ARCHITEKT (V5.1): Koordiniert das Deployment aller Agenten.
    Verankert Rollen, Hardware-Limits und Sicherheit-Einstellungen
    dauerhaft in der Registry (BEHAVIOR_DIR).
    """
    # 1. STRUKTUR-SICHERUNG: Prüft, ob die Intelligence-Registry existiert
    if not os.path.exists(BEHAVIOR_DIR):
        os.makedirs(BEHAVIOR_DIR, exist_ok=True)

    # 2. AUSFÜHRUNG DER EINZEL-MUSTER (Modularer Aufruf)
    try:
        deploy_agent_code_python()
        deploy_agent_text_datascientist()
        deploy_agent_plot_statistic()

    except Exception as e:
        print(f"❌ Fehler bei Deployment: {e}")

# INITIALISIERUNG
# installiert die Agenten-Intelligenz physisch auf dem MacBook
deploy_specialist_agents_yaml()


In [7]:
# LLM-AGENTEN-STEUERUNG (ADAPTER  - MODULARER AUFBAU)

# 1. KOMPONENTE: VERBINDUNGS-WÄCHTER (Auto-Check/Start)
def pruefe_und_starte_ollama():
    url = "http://localhost:11434"
    # TEST: Ist es da?
    try:
        if requests.get(url, timeout=1).status_code == 200: return True
    except: pass

    # NICHT DA -> SOFORT REPARIEREN
    try:
        if sys.platform == "darwin":
            subprocess.Popen(["open", "-a", "Ollama"])
        elif sys.platform == "win32":
            p = os.path.expandvars(r"%LocalAppData%\Ollama\ollama app.exe")
            subprocess.Popen([p] if os.path.exists(p) else ["ollama", "serve"])
            if os.path.exists(p):
                subprocess.Popen([p])
            else:
                subprocess.Popen(["ollama", "serve"], shell=True)
        # 10 Sek. Polling: Wir warten aktiv auf den Erfolg
        for _ in range(10):
            time.sleep(1)
            try:
                if requests.get(url, timeout=1).status_code == 200:
                    return True
            except: continue
    except Exception as e:
        print(f"⚠️ Fehler beim Startversuch: {e}")

    # SCHEITERT ALLES -> DEINE ANWEISUNG
    print("\n🛑 START FEHLGESCHLAGEN. Nächste Schritte:")
    print("1. Installieren (ollama.com)\n2. App manuell starten\n3. Port 11434 prüfen")
    return False

# 2. KOMPONENTE: SYSTEM-CHECK
def perform_system_check():
    ram_avail = psutil.virtual_memory().available / (1024**3)
    if ram_avail < 1.2:
        gc.collect()
        return f"⚠️ RAM kritisch ({ram_avail:.2f}GB). GC ausgeführt."
    return ""#f"✅ RAM OK ({ram_avail:.2f}GB)."

# 3. KOMPONENTE: INTUITIONS-CHECK & VEREDELUNG
def veredle_input(frage):
    f_up = frage.upper()
    veredelt = frage
    # Falls Benutzer ungenau ist, helfe dem Agenten mit Spalten-Wichtigkeit
    if "STIMMUNG" in f_up or "SENTIMENT" in f_up:
        veredelt += " (Nutze bevorzugt die Spalte 'sentiment' für die Analyse)."
    if "ZEIT" in f_up or "VERLAUF" in f_up:
        veredelt += " (Prüfe die Spalte 'date' für Zeitreihen-Plots)."
    return veredelt

# 4. KOMPONENTE: PROGRESS-ANIMATION
def start_spinner(stop_event, is_de=True):
    chars = ['|', '/', '-', '\\']
    msg = "Analysiere (Lokale GPU)..." if is_de else "Analyzing (Local GPU)..."
    idx = 0
    while not stop_event.is_set():
        sys.stdout.write(f'\r{msg} {chars[idx % 4]} ')
        sys.stdout.flush()
        idx += 1
        time.sleep(0.1)
    sys.stdout.write('\r' + ' ' * 50 + '\r')

# 5. KOMPONENTE: ADAPTER-KLASSE (Gott-Klasse V13.1 - Finaler Fix)
class OllamaAgentOrchestrator(LLM):
    def __init__(self, model_name: str = "llama3"):
        super().__init__()
        self.model_name = model_name

    @property
    def type(self) -> str: return "ollama_pandasai"

    def call(self, instruction: str, ram_facts: dict = None, **kwargs) -> str:
        """
        DER NEUTRALE VERWALTER (Gott-Klasse)
        Nimmt die ram_facts diskret entgegen und steuert die Experten-Wahl.
        """
        original_query = str(instruction)
        f_up = original_query.upper()

        # Strategische Weichenstellung
        code_keywords = ["MERKMALE", "DATENTYPEN", "FEHLENDE", "SUMMARY", "STATISTIK"]
        is_code_task = any(x in f_up for x in code_keywords) or "[TASK: CODE]" in f_up
        is_plot_task = "[TASK: PLOT]" in f_up
        is_eda_task = "[TASK: EDA]" in f_up

        if is_code_task or is_plot_task or is_eda_task:
            return self.generate_code(original_query, ram_facts=ram_facts, **kwargs)

        return self.generate(original_query, ram_facts=ram_facts, **kwargs)

    def call_internal(self, prompt: str) -> str:
        payload = {"model": self.model_name, "prompt": prompt, "stream": False, "options": {"temperature": 0.0}}
        try:
            res = requests.post("http://localhost:11434/api/generate", json=payload, timeout=60)
            return res.json().get("response", "").strip().upper()
        except: return "TEXT_DATASCIENTIST"

    def generate(self, prompt: str, ram_facts: dict = None, **kwargs) -> str:
        # 1. SCAN & ROUTING (Spezialisten-Wahl)
        agent_files = list(Path(BEHAVIOR_DIR).glob("*.yaml"))
        agent_catalog = []
        for f in agent_files:
            try:
                with open(f, 'r', encoding='utf-8') as s:
                    cfg = yaml.safe_load(s)
                    agent_catalog.append({"name": cfg.get('agent_name'), "file": f})
            except: continue

        routing_prompt = f"Wähle Experte für: '{prompt}' aus {[a['name'] for a in agent_catalog]}. Antworte NUR mit dem Namen."
        wahl = self.call_internal(routing_prompt)
        selected = next((a for a in agent_catalog if a['name'] in wahl), agent_catalog[0])

        with open(selected['file'], 'r', encoding='utf-8') as s:
            dna = yaml.safe_load(s)

        # --- DYNAMISCHE SPRACH-ERKENNUNG (Chef-Vorgabe) ---
        target_lang = kwargs.get("Language", "DE").upper()
        if target_lang == "DE":
            sprach_regel = "ANTWORTE STRENG AUF DEUTSCH. NUTZE DEUTSCHE FACHBEGRIFFE."
        else:
            sprach_regel = f"ANSWER STRICTLY IN {target_lang}."

        # 2. DAS GOLDENE MANDAT (Interne System-Hygienisierung)
        internal_mandate = (
            f"IDENTITÄT: Du handelst als {selected['name']}. Rolle: {dna.get('role', 'Experte')}.\n"
            f"SPRACHE: {sprach_regel}\n"
            f"MANDAT: Nutze TEMP_Clear für Korrekturen, Automated EDA für Struktur.\n"
            f"HARDWARE: 1 Core Reserve Standard [2025-12-29] strikt einhalten.\n"
            f"DATEN-INTEGRITÄT: Zitiere nur gelieferte RAM_FACTS. Raten ist untersagt."
            f"TABELLEN-PFLICHT: Alle numerischen Merkmale (Counts, Nulls) müssen in einer Markdown-Tabelle stehen. Kein reiner Fließtext!"
        )

        # 3. EXECUTION (Fakten-Injektion & Prompt-Kapselung)
        df_name = ram_facts.get('df_name', 'Daten') if ram_facts else "UNBEKANNT"

        payload = {
            "model": self.model_name,
            "prompt": (
                f"[SYSTEM_INTERNAL: {internal_mandate}]\n"
                f"[DNA_INSTRUCTIONS: {dna.get('instructions')}]\n"
                f"[RAM_SNAPSHOT: {ram_facts}]\n\n"
                f"### ANALYSE-AUFTRAG: {prompt}\n"
                f"### FINALE REGEL: {sprach_regel} KEIN SMALLTALK. DIREKTER START FÜR {df_name}:"
            ),
            "stream": False,
            "options": {"temperature": 0.0}
        }
        try:
            print(f"🧬 Spezialist '{selected['name']}' dokumentiert {df_name}...")
            # Zeitfenster für den API-Ruf (Gesamt-Sicherheit)
            res = requests.post("http://localhost:11434/api/generate", json=payload, timeout=2000)
            output = res.json().get("response", "").strip()

            # --- DER TÜRRIEGEL (Hausmeister-Fix) ---
            # Wir prüfen das Signal. Alles, was danach kommt, ist der "Mörder-Klon".
            if "### AGENT_PROCESS_COMPLETE ###" in output:
                # Der hinterlistige Angriff wird hier im Keim erstickt
                output = output.split("### AGENT_PROCESS_COMPLETE ###")[0].strip()
                # print(f"✅ Prozess-Signal 'COMPLETE' für {df_name} empfangen.")
            else:
                print(f"⚠️ WARNUNG: Agent '{selected['name']}' unvollständig (Signal fehlt).")

            # --- REINIGUNG (Hygiene & Signal-Ausblendung) ---
            # 1. System-Tags entfernen
            tags_to_remove = ["[SYSTEM_INTERNAL", "### SYSTEM-REGEL", "DNA_INSTRUCTIONS", "RAM_SNAPSHOT"]
            for tag in tags_to_remove:
                if tag in output[:150]:
                    pos = output.find("]:")
                    if pos != -1:
                        output = output[pos+2:].strip()
                    elif "###" in output[:50]:
                        output = output.split(":")[-1].strip() if ":" in output else output

            # 2. Prozess-Signale vor dem Nutzer verbergen [2025-10-06]
            signals_to_hide = ["### PHASE: THINKING_READY ###"]
            for sig in signals_to_hide:
                output = output.replace(sig, "").strip()

            # 3. DER ECHO-KILLER (Verteidigung gegen den messernden Agenten)
            # Falls der Agent sich innerhalb des Textes wiederholt:
            lines = output.split('\n')
            unique_lines = []
            seen_content = set()
            for line in lines:
                stripped = line.strip().lower()
                if stripped not in seen_content or len(stripped) < 10: # Kurze Zeilen/Umbruche erlauben
                    unique_lines.append(line)
                    if len(stripped) > 20: # Nur signifikante Sätze tracken
                        seen_content.add(stripped)
            output = '\n'.join(unique_lines).strip()

            # 4. Persona-Reste entfernen
            #if "Senior Data Scientist" in output[:100]:
            #    lines = output.split('\n')
            #    output = '\n'.join(lines[1:]).strip() if len(lines) > 1 else output

            gc.collect()
            return output

        except Exception as e:
            return f"❌ Fehler im Verwalter-Dienst ({selected['name']}): {e}"

    def generate_code(self, prompt: str, ram_facts: dict = None, **kwargs) -> str:
        """Extrahiert das ausführbare Muster (Code)."""
        raw_response = self.generate(prompt, ram_facts=ram_facts, **kwargs)
        code_match = re.search(r'```python\s*(.*?)\s*```', raw_response, re.DOTALL)
        logic_code = code_match.group(1).strip() if code_match else raw_response.strip()

        # Hardware-Schutz: MacBook Air Plot Fix
        if "plt." in logic_code and "plt.savefig" not in logic_code:
            logic_code += f"\nplt.savefig('exports/plots/auto_plot.png')\nprint('PLOT_SAVED')"
        return logic_code

# 8. KOMPONENTE: UNIVERSAL-STEUERUNG
def frage(text_user, Language="DE"):
    """
    UNIVERSALER STRATEGISCHER VEREDLER
    - Trennt interne Mandate von der text_user.
    - Extrahiert harte RAM-Fakten zur Vermeidung von Halluzinationen.
    """
    global llm

    if not pruefe_und_starte_ollama(): return

    # System-Check (RAM-Schutz [2025-11-21])
    check_msg = perform_system_check()
    if check_msg: print(check_msg)

    start_time = time.time()

    # Fortschrittsanzeige (Thread-basiert)
    stop_event = threading.Event()
    t = threading.Thread(target=start_spinner, args=(stop_event, Language=="DE"))
    t.start()

    try:
        # 1. DYNAMISCHE DATEN-IDENTIFIKATION
        # Priorität: df_Cleaning -> df_ML -> Erstbestes df_...
        aktueller_df_name = "df_Cleaning" if "df_Cleaning" in globals() else \
                           ("df_ML" if "df_ML" in globals() else \
                           (next((n for n in globals() if n.startswith("df_")), "df")))

        df_ref = globals().get(aktueller_df_name)

        # 2. METADATEN-EXTRAKTION (Wahrheit aus dem RAM)
        ram_facts = {}
        if df_ref is not None:
            ram_facts = {
                "df_name": aktueller_df_name,
                "dtypes": df_ref.dtypes.astype(str).to_dict(),
                "nans": df_ref.isnull().sum().to_dict(),
                "rows": len(df_ref),
                "cols": list(df_ref.columns),
                "stats_sample": df_ref.describe(include='all').iloc[0:3].to_dict()
            }

        # 3. UNIVERSAL-MAPPING & SPALTEN-INTEGRITÄT
        veredelter_text = text_user.lower()
        f_up = text_user.upper()

        # Erkennt Spaltennamen im Text, um dem Agenten den Fokus zu erleichtern
        gefundene_spalten = [s for s in ram_facts.get("cols", []) if s.lower() in veredelter_text]
        spalten_context = f" (Fokus auf Spalten: {gefundene_spalten})" if gefundene_spalten else ""

        # 4. TASK-MATRIX (Routing-Präferenz für den Orchestrator)
        if any(k in f_up for k in ["PLOT", "GRAFIK", "CHART", "VISUALISIER"]):
            task_prefix = "[TASK: PLOT]"
        elif any(k in f_up for k in ["CODE", "MUSTER", "BERECHN", "REINIG", "TRANSFORM"]):
            task_prefix = "[TASK: CODE]"
        elif any(k in f_up for k in ["WAS", "WARUM", "ANALYSIER", "BESCHREIB", "ZUSAMMENFASS", "SUMMARY"]):
            task_prefix = "[TASK: EDA]"
        else:
            task_prefix = "[TASK: ANALYSIS]"

        # 5. DER SAUBERE AUFTRAG
        optimized_query = f"{task_prefix} {veredelter_text}{spalten_context}"

        # 6. RUF AN VERWALTER MIT DISKRETEN FAKTEN
        # Hier findet die Experten-Wahl und DNA-Aktivierung statt
        antwort = llm.call(optimized_query, ram_facts=ram_facts, Language=Language)

        # 7. AUSGABE-FIX (Sichtbarkeit wiederhergestellt)
        stop_event.set()
        t.join()

        dur = round(time.time() - start_time, 1)

        # Finaler Output-Block für das Notebook
        print(f"\nANTWORT ({dur}s):")
        print(antwort)

        return None

    except Exception as e:
        if 'stop_event' in locals(): stop_event.set()
        if 't' in locals(): t.join()
        print(f"❌ Fehler in der Universal-Steuerung: {e}")
        return None

# Initialisierung des Adapters
llm = OllamaAgentOrchestrator(model_name="llama3")

1. Nutzen Sie LLM für erste Erkundungen
Laden den Datensatz, nutzen den integrierten KI-Assistenten LLM, um Ihr Projekt zu starten. Bitten Sie LLM, den Datensatz zu beschreiben und wichtige Merkmale wie Spaltennamen, Datentypen und fehlende Werte zusammenzufassen.
Diese automatisierte Analyse hilft Ihnen, die Struktur des Datensatzes schnell zu erfassen und zu entscheiden, welche Spalten für Ihre Ziele am relevantesten sind.

In [8]:
frage("Datensatz beschreiben in dem die wichtige Merkmale wie Spaltennamen, Datentypen und fehlende Werte zusammenzufassen")

Analysiere (Lokale GPU)... \ 🧬 Spezialist 'TEXT_DATASCIENTIST' dokumentiert df_sample...
                                                  
ANTWORT (110.3s):
### DATEN-PROTOKOLL: df_sample



Die Datenlage in df_sample belegt, dass es sich um einen Datensatz mit 93 Zeilen und 7 Spalten handelt. Die wichtigsten Merkmale sind:

| Spalte | Datentyp | Fehlende Werte | Eindeutige Werte | Kardinalität % |
| --- | --- | --- | --- | --- |
| tweet_id | int64 | 0 | - | 100% |
| author_id | object | 0 | - | 45.2% |
| inbound | bool | 0 | True/False | 50% |
| created_at | object | 0 | - | 100% |
| text | object | 0 | - | 100% |
| response_tweet_id | object | 28 | - | 70.1% |
| in_response_to_tweet_id | float64 | 25 | - | 27.4% |



Die Kardinalität der Spalte "tweet_id" beträgt 100%, die Saisonalität ist nicht eindeutig zu bestimmen, da keine zeitlichen Abhängigkeiten erkennbar sind. Die Füllrate aller Spalten liegt bei 100%. Es gibt keine Skewness in den Daten.


2. Visualisieren Sie die Daten mit AutoViz.
Als Nächstes auf das visuelle Verständnis der Daten. Mit AutoViz können Diagramme und Grafiken erstellen, die wichtige Erkenntnisse über die Daten liefern. Vergleichen die Ergebnisse von AutoViz mit den vorherigen Vorschlägen von LLM.

In [9]:
frage('Als Nächstes auf das visuelle Verständnis der Daten. Mit AutoViz können Diagramme und Grafiken erstellen, die wichtige Erkenntnisse über die Daten liefern. Vergleichen die Ergebnisse von AutoViz mit den vorherigen Vorschlägen von LLM.')

Analysiere (Lokale GPU)... \ 🧬 Spezialist 'TEXT_DATASCIENTIST' dokumentiert df_sample...
                                                  
ANTWORT (138.8s):
### DATEN-PROTOKOLL: df_sample



Die Datenlage in df_sample belegt, dass es sich um eine Tabelle mit 93 Zeilen und 7 Spalten handelt. Die Spaltennamen sind tweet_id, author_id, inbound, created_at, text, response_tweet_id und in_response_to_tweet_id.

### BEWEISAUFNAHME (Tabelle):

| Spalte | Datentyp | Fehlende Werte | Eindeutige Werte | Kardinalität % |
| --- | --- | --- | --- | --- |
| tweet_id | int64 | 0 | - | 100% |
| author_id | object | 0 | - | 45.2% |
| inbound | bool | 0 | True/False | 50% |
| created_at | object | 0 | - | 100% |
| text | object | 0 | - | 100% |
| response_tweet_id | object | 28 | - | 70.1% |
| in_response_to_tweet_id | float64 | 25 | - | 27.4% |

### ANALYSE:

Die Kardinalität der Spalte tweet_id beträgt 100%, was darauf hindeutet, dass es sich um eine eindeutige Kennziffer handelt. Die Spalte author_i

3. Stimmungsanalyse mithilfe von Hugging Face LLMs
Wenden abschließend ein Large Language Model (LLM) von Hugging Face an, um die textSpalte des Datensatzes zu analysieren. Diese Spalte enthält den vollständigen Tweet des Kunden. Mithilfe des Modells bewerten Sie die Stimmung der Nachrichten.
Sie können entweder Folgendes verwenden:
- Die Hugging Face Transformers-Bibliothek oder
- die Inference API ermöglicht den Zugriff, ohne Modelle herunterladen zu müssen.

# Präsentationskontext
Dieses Projekt wird im Rahmen einer Live-Präsentation vorgestellt, nachdem alle drei Projekte zur Steigerung der KI-gestützten Produktivität abgeschlossen sind.
Im Rahmen der Präsentation werden Sie Folgendes erläutern:
- Das analytische Ziel dieses Projekts
- Wie Gemini, AutoViz und Sprachmodelle verwendet wurden
- Wichtige Erkenntnisse zur Stimmungslage aufgedeckt
- Was Sie über die Verwendung von KI für textbasierte Analysen gelernt haben

Der Schwerpunkt der Präsentation liegt auf der Interpretation und der Steigerung der Produktivität, nicht auf technischen Details der Umsetzung.

# Qualitätserwartungen
- KI-Werkzeuge sollten gezielt und nicht oberflächlich eingesetzt werden.
- Die Ergebnisse müssen klar interpretiert und in den Kontext gesetzt werden.
- Die Reflexionen sollten ein Verständnis sowohl der Daten als auch der Werkzeuge erkennen lassen.
- Die gewonnenen Erkenntnisse sollten auf den im Rahmen der Analyse gewonnenen Erkenntnissen beruhen.